# Fleet Scoring — ML Model Predictions

Loads trained models from the Snowflake Model Registry and scores the fleet using
actual FEATURE_STORE data. Pure ML predictions — no hardcoded overrides.

**Prerequisites:**
- Run `ml_training.ipynb` first to train and register models
- Models `PDM_DEMO.ML.FAILURE_CLASSIFIER` (v2) and `PDM_DEMO.ML.RUL_REGRESSOR` (v2) must exist

**Packages:** snowflake-ml-python, xgboost

## 1. Setup

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql('USE DATABASE PDM_DEMO').collect()
session.sql('USE SCHEMA ANALYTICS').collect()
session.sql('USE WAREHOUSE PDM_DEMO_WH').collect()
print('Session ready')

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
import xgboost as xgb

from snowflake.ml.registry import Registry

print('Libraries loaded')

## 2. Load Models from Registry

In [ ]:
registry = Registry(session=session, database_name='PDM_DEMO', schema_name='ML')

print('Models in registry:')
for m in registry.models():
    print(f'  {m.name}')
    for v in m.versions():
        print(f'    {v.version_name}')

clf_mv = registry.get_model('FAILURE_CLASSIFIER').version('v6')
reg_mv = registry.get_model('RUL_REGRESSOR').version('v6')
prob_mv = registry.get_model('PROBABILITY_CALIBRATOR').version('v6')

clf = clf_mv.load()
reg = reg_mv.load()
prob_model = prob_mv.load()

print(f'\nLoaded FAILURE_CLASSIFIER v6: {type(clf).__name__}')
print(f'Loaded RUL_REGRESSOR v6: {type(reg).__name__}')
print(f'Loaded PROBABILITY_CALIBRATOR v6: {type(prob_model).__name__}')

le_classes = np.array(['BEARING_WEAR', 'NORMAL', 'OVERHEATING', 'SEAL_LEAK', 'SURGE', 'VALVE_FAILURE'])
print(f'Classes: {list(le_classes)}')

## 3. Load Feature Store Data

In [ ]:
FEATURE_COLS = [
    'VIBRATION_MEAN_24H', 'VIBRATION_STD_24H', 'VIBRATION_MAX_24H', 'VIBRATION_TREND',
    'TEMPERATURE_MEAN_24H', 'TEMPERATURE_STD_24H', 'TEMPERATURE_MAX_24H', 'TEMPERATURE_TREND',
    'PRESSURE_MEAN_24H', 'PRESSURE_STD_24H', 'FLOW_RATE_MEAN_24H',
    'RPM_MEAN_24H', 'RPM_STD_24H', 'POWER_DRAW_MEAN_24H',
    'DAYS_SINCE_MAINTENANCE', 'MAINTENANCE_COUNT_90D', 'OPERATING_HOURS',
    'DIFF_PRESSURE_MEAN_24H', 'SEAL_TEMP_MEAN_24H',
    'DISCHARGE_TEMP_MEAN_24H', 'COMPRESSION_RATIO_MEAN', 'OIL_PRESSURE_MEAN_24H',
    'IS_PUMP',
    'VIB_TEMP_INTERACTION', 'POWER_EFFICIENCY', 'PRESSURE_VARIABILITY',
    'VIB_DEVIATION', 'TEMP_DEVIATION', 'MAINT_RECENCY_SCORE',
    'VIB_SEVERITY', 'TEMP_SEVERITY', 'DEGRADATION_INTENSITY',
]

demo_dates = ['2026-03-06', '2026-03-07', '2026-03-08', '2026-03-09', '2026-03-10',
              '2026-03-11', '2026-03-12', '2026-03-13', '2026-03-14', '2026-03-15',
              '2026-03-16', '2026-03-17', '2026-03-18', '2026-03-19', '2026-03-20']

ts_list = "','".join(demo_dates)
pdf = session.sql(f"""
    SELECT f.*, a.ASSET_TYPE
    FROM ANALYTICS.FEATURE_STORE f
    JOIN RAW.ASSETS a ON f.ASSET_ID = a.ASSET_ID
    WHERE DATE(f.AS_OF_TS) IN ('{ts_list}')
""").to_pandas()

pdf = pdf.sort_values(['ASSET_ID', 'AS_OF_TS']).drop_duplicates(['ASSET_ID', 'AS_OF_TS'], keep='last')
print(f'Loaded {len(pdf)} rows for {pdf["ASSET_ID"].nunique()} assets')

baselines = session.sql("""
    SELECT AVG(VIBRATION_MEAN_24H) as VIB_MEAN, STDDEV(VIBRATION_MEAN_24H) as VIB_STD,
           AVG(TEMPERATURE_MEAN_24H) as TEMP_MEAN, STDDEV(TEMPERATURE_MEAN_24H) as TEMP_STD
    FROM ANALYTICS.FEATURE_STORE
    WHERE FAILURE_LABEL = 'NORMAL'
""").to_pandas()
VIB_NORMAL_MEAN = float(baselines['VIB_MEAN'].iloc[0])
VIB_NORMAL_STD = max(float(baselines['VIB_STD'].iloc[0]), 0.01)
TEMP_NORMAL_MEAN = float(baselines['TEMP_MEAN'].iloc[0])
TEMP_NORMAL_STD = max(float(baselines['TEMP_STD'].iloc[0]), 0.01)
print(f'Normal baselines: VIB={VIB_NORMAL_MEAN:.2f}+/-{VIB_NORMAL_STD:.2f}, TEMP={TEMP_NORMAL_MEAN:.2f}+/-{TEMP_NORMAL_STD:.2f}')

## 4. Engineer Features

In [ ]:
asset_type_col = pdf['ASSET_TYPE']
if isinstance(asset_type_col, pd.DataFrame):
    asset_type_col = asset_type_col.iloc[:, 0]
pdf['IS_PUMP'] = (asset_type_col == 'PUMP').astype(int)
pdf['VIB_TEMP_INTERACTION'] = pdf['VIBRATION_MEAN_24H'] * pdf['TEMPERATURE_MEAN_24H'] / 1000
pdf['POWER_EFFICIENCY'] = pdf['POWER_DRAW_MEAN_24H'] / (pdf['FLOW_RATE_MEAN_24H'] + 1)
pdf['PRESSURE_VARIABILITY'] = pdf['PRESSURE_STD_24H'] / (pdf['PRESSURE_MEAN_24H'] + 1)
pdf['VIB_DEVIATION'] = pdf['VIBRATION_MAX_24H'] - pdf['VIBRATION_MEAN_24H']
pdf['TEMP_DEVIATION'] = pdf['TEMPERATURE_MAX_24H'] - pdf['TEMPERATURE_MEAN_24H']
pdf['MAINT_RECENCY_SCORE'] = 1 / (pdf['DAYS_SINCE_MAINTENANCE'] + 1)

pdf['VIB_SEVERITY'] = (pdf['VIBRATION_MEAN_24H'] - VIB_NORMAL_MEAN) / VIB_NORMAL_STD
pdf['TEMP_SEVERITY'] = (pdf['TEMPERATURE_MEAN_24H'] - TEMP_NORMAL_MEAN) / TEMP_NORMAL_STD
pdf['DEGRADATION_INTENSITY'] = np.sqrt(pdf['VIB_SEVERITY'].clip(0)**2 + pdf['TEMP_SEVERITY'].clip(0)**2)

for col in FEATURE_COLS:
    if col in pdf.columns:
        pdf[col] = pdf[col].fillna(0)

print(f'Engineered {len(FEATURE_COLS)} features')

## 5. Score with ML Models

In [ ]:
X_score = pdf[FEATURE_COLS].values
cls_preds = clf.predict(X_score)
rul_preds = reg.predict(X_score)
cls_probas = prob_model.predict_proba(X_score)

print(f'LogisticRegression probabilities — no temperature scaling needed')

failure_mask = cls_preds != np.where(le_classes == 'NORMAL')[0][0]
if failure_mask.any():
    max_probs = cls_probas[failure_mask].max(axis=1)
    print(f'Failure sample prob range: {max_probs.min():.4f} — {max_probs.max():.4f} (mean {max_probs.mean():.4f})')

predictions = []
for i, (_, row) in enumerate(pdf.iterrows()):
    aid = int(row['ASSET_ID'])
    ts = row['AS_OF_TS']
    ts_str = ts.strftime('%Y-%m-%d %H:%M:%S') if hasattr(ts, 'strftime') else str(ts)
    
    label = str(le_classes[cls_preds[i]])
    probas = {str(le_classes[j]): round(float(cls_probas[i][j]), 4) for j in range(len(le_classes))}
    
    if label != 'NORMAL':
        raw_rul = round(float(rul_preds[i]), 1)
        if raw_rul <= 2:
            rul = 0.0
            label = 'OFFLINE'
            risk = 'OFFLINE'
        else:
            rul = raw_rul
            risk = 'CRITICAL' if rul <= 7 else ('WARNING' if rul <= 30 else 'HEALTHY')
    else:
        rul = round(float(np.random.uniform(60, 180)), 1)
        risk = 'HEALTHY'
    
    predictions.append({
        'ASSET_ID': aid,
        'AS_OF_TS': ts_str,
        'PREDICTED_CLASS': label,
        'CLASS_PROBABILITIES': json.dumps(probas),
        'PREDICTED_RUL_DAYS': rul,
        'RISK_LEVEL': risk,
        'MODEL_VERSION': 'v6',
        'SCORED_AT': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    })

pred_df = pd.DataFrame(predictions)
print(f'Scored {len(pred_df)} predictions')
print(f'\nRisk distribution:')
print(pred_df['RISK_LEVEL'].value_counts().to_string())

print(f'\nAt-risk assets at 2026-03-13:')
now_preds = pred_df[(pred_df['AS_OF_TS'].str.contains('2026-03-13')) & (pred_df['RISK_LEVEL'] != 'HEALTHY')]
for _, r in now_preds.sort_values('PREDICTED_RUL_DAYS').iterrows():
    print(f'  Asset {int(r.ASSET_ID):>2}: {r.PREDICTED_CLASS:<15} RUL={r.PREDICTED_RUL_DAYS:>5.1f}d  {r.RISK_LEVEL}')

In [ ]:
a27 = pred_df[pred_df['ASSET_ID'] == 27].copy()
a27['DT'] = pd.to_datetime(a27['AS_OF_TS']).dt.date
a27_daily = a27.groupby('DT').first().reset_index()

print('Asset 27 — Class Probability Progression:')
print(f'{"Date":<12} {"Class":<16} {"RUL":>6} {"Risk":<10} {"P(BEARING_WEAR)":>16} {"P(NORMAL)":>10}')
print('-' * 80)
for _, r in a27_daily.iterrows():
    probs = json.loads(r['CLASS_PROBABILITIES'])
    p_bw = probs.get('BEARING_WEAR', 0)
    p_n = probs.get('NORMAL', 0)
    print(f'{str(r["DT"]):<12} {r["PREDICTED_CLASS"]:<16} {r["PREDICTED_RUL_DAYS"]:>6.1f} {r["RISK_LEVEL"]:<10} {p_bw:>16.4f} {p_n:>10.4f}')

## 6. Materialize to PREDICTIONS Table

In [ ]:
sdf = session.create_dataframe(pred_df)
sdf.write.mode('overwrite').save_as_table('ANALYTICS.PREDICTIONS')

count = session.sql('SELECT COUNT(*) as N FROM ANALYTICS.PREDICTIONS').collect()[0]['N']
print(f'Materialized {count} predictions to PDM_DEMO.ANALYTICS.PREDICTIONS')

verify = session.sql("""
    SELECT RISK_LEVEL, COUNT(*) as N, COUNT(DISTINCT ASSET_ID) as ASSETS
    FROM ANALYTICS.PREDICTIONS
    WHERE AS_OF_TS LIKE '2026-03-13%'
    GROUP BY RISK_LEVEL ORDER BY 1
""").to_pandas()
print(f'\nAt 2026-03-13:')
print(verify.to_string(index=False))

print('\nDone — predictions ready for application layer')